In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import open3d as o3d
from PIL import Image
from scipy.interpolate import griddata

def get_cuboid_mesh(verts, elems, values, cmap="viridis"):
    tri_idx = np.array([[0,0,0,0,0,0,6,6,6,6,6,6],
                        [2,3,7,4,1,5,2,3,1,5,4,7],
                        [1,2,3,7,5,4,3,7,2,1,5,4]], dtype=int)
    mesh = o3d.geometry.TriangleMesh()
    tris = np.zeros([12*elems.shape[0],3], dtype=int)
    for i, elem in enumerate(elems):
        tris[i*12:(i+1)*12,:] = elem[tri_idx].T-1
    
    mesh.vertices = o3d.utility.Vector3dVector(verts)
    mesh.triangles = o3d.utility.Vector3iVector(tris)

    # Subdivide the mesh to increase point density
    mesh = mesh.subdivide_midpoint(number_of_iterations=1)

    # Interpolate stress values for new vertices
    old_vertices = np.asarray(verts)
    new_vertices = np.asarray(mesh.vertices)
    
    interpolated_values = griddata(old_vertices, values, new_vertices, method='linear', fill_value=np.mean(values))

    # Normalize the interpolated values
    normalized_values = (interpolated_values - np.min(values)) / (np.max(values) - np.min(values))
    
    # Apply colormap
    cmap = plt.get_cmap(cmap)
    colors = cmap(normalized_values)[:,:3]
    
    mesh.vertex_colors = o3d.utility.Vector3dVector(colors)

    # Apply Taubin smoothing
    mesh = mesh.filter_smooth_taubin(number_of_iterations=5)

    return mesh

data_path = r"C:\Users\XuanLiang\Documents\Netfabb_Fusion360_data\TEST\\data\vmstress\\"
size = os.listdir(data_path)
data_files = os.listdir(data_path)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:

def rotate_shape(mesh, angle_x=0, angle_y=0, angle_z=0):
    # Create rotation matrices
    R_x = np.array([[1, 0, 0],
                    [0, np.cos(angle_x), -np.sin(angle_x)],
                    [0, np.sin(angle_x), np.cos(angle_x)]])

    R_y = np.array([[np.cos(angle_y), 0, np.sin(angle_y)],
                    [0, 1, 0],
                    [-np.sin(angle_y), 0, np.cos(angle_y)]])

    R_z = np.array([[np.cos(angle_z), -np.sin(angle_z), 0],
                    [np.sin(angle_z), np.cos(angle_z), 0],
                    [0, 0, 1]])

    # Combine rotations
    R = R_z @ R_y @ R_x

    # Convert to Open3D rotation matrix
    rotation = o3d.geometry.get_rotation_matrix_from_xyz((angle_x, angle_y, angle_z))

    # Rotate the mesh
    mesh.rotate(rotation, center=mesh.get_center())
    
    return mesh

def modify_FOV(vis):
    screenshot = vis.capture_screen_float_buffer(False)
    # Convert the NumPy array to an image
    img = Image.fromarray((np.asarray(screenshot) * 255).astype('uint8'), 'RGB')
    filepath = os.path.join(directory, f'part{i}_position{j}.jpg')
    img.save(filepath)
    vis.destroy_window()

for i in range(len(size)-1):
    directory = r"C:\Users\XuanLiang\Documents\Netfabb_Fusion360_data\TEST\data\vmstress\images"
    data = np.load(data_path + data_files[i])
    verts, elems, stress = data["verts"], data["elems"], data["vmstress"]
    mesh = get_cuboid_mesh(verts, elems, stress[:,0])
    for j in range(12):
        vis = o3d.visualization.Visualizer()
        vis.create_window()
        vis.add_geometry(mesh)

        angle = j * np.pi / 3  # Rotate by 60 degrees each time
        mesh = rotate_shape(mesh, angle_y=angle, angle_x=-np.pi/6)
        vis.update_geometry(mesh)
        vis.poll_events()
        vis.update_renderer()

        vis.register_animation_callback(modify_FOV)

        vis.run()
        vis.destroy_window()